# Day 30 — Power BI Data Model
Project 1 | Prepare a star schema. Customer names are proxies; technical keys do not establish identity. Product keys represent product/category combinations. Subcategory is absent; city stays in the fact table. Profit is unknown.

In [1]:
import pandas as pd
from IPython.display import display
s = pd.read_csv('transformed_retail_sales.csv', parse_dates=['Order_Date'])
# Deterministic technical keys, not original business identifiers.
c = s[['Customer_Name','Customer_Segment']].drop_duplicates().sort_values('Customer_Name').reset_index(drop=True)
assert c.Customer_Name.is_unique
c.insert(0,'Customer_Key',range(1,len(c)+1))
# City is retained on transactions because it may vary by customer.
q = s[['Product','Category']].drop_duplicates().sort_values(['Product','Category']).reset_index(drop=True)
q.insert(0,'Product_Key',range(1,len(q)+1))
d = pd.DataFrame({'Date':pd.date_range(s.Order_Date.min().replace(month=1,day=1),s.Order_Date.max().replace(month=12,day=31))})
d['Year']=d.Date.dt.year
d['Month']=d.Date.dt.month_name()
d['Month_Number']=d.Date.dt.month
d['Quarter']=d.Date.dt.quarter
f=s.merge(c,on=['Customer_Name','Customer_Segment'],validate='many_to_one').merge(q,on=['Product','Category'],validate='many_to_one')
f=f[['Order_ID','Customer_Key','Product_Key','Order_Date','City','Quantity','Unit_Price','Sales','Customer_Rating']].copy()
f['Profit']=float('nan')
q=q.rename(columns={'Product':'Product_Name'})
for name,t in [('Fact_Sales',f),('Dim_Customer',c),('Dim_Product',q),('Dim_Date',d)]:
    t.to_csv(name+'.csv',index=False,date_format='%Y-%m-%d')
    print(name, t.shape)
    display(t.head())


Fact_Sales (30, 10)


,Order_ID,Customer_Key,Product_Key,Order_Date,City,Quantity,Unit_Price,Sales,Customer_Rating,Profit
0,ORD001,5,4,2026-10-02,Mumbai,2,650,1300,5.0,NaN
1,ORD002,3,3,2026-10-03,Nagpur,3,320,960,3.0,NaN
2,ORD003,6,2,2026-10-04,Nashik,4,1100,4400,4.0,NaN
3,ORD004,4,1,2026-10-05,Pune,5,1500,7500,5.0,NaN
4,ORD005,1,4,2026-10-06,Pune,1,650,650,4.0,NaN


Dim_Customer (6, 3)


,Customer_Key,Customer_Name,Customer_Segment
0,1,Arjun,Medium Value
1,2,Asha,Medium Value
2,3,Neha,Medium Value
3,4,Priya,High Value
4,5,Rahul,Medium Value


Dim_Product (4, 3)


,Product_Key,Product_Name,Category
0,1,Headphones,Electronics
1,2,Lamp,Home
2,3,Rice,Grocery
3,4,T-Shirt,Clothing


Dim_Date (365, 5)


,Date,Year,Month,Month_Number,Quarter
0,2026-01-01,2026,January,1,1
1,2026-01-02,2026,January,1,1
2,2026-01-03,2026,January,1,1
3,2026-01-04,2026,January,1,1
4,2026-01-05,2026,January,1,1


## Validate keys and reconcile totals

In [2]:
assert c.Customer_Key.is_unique and q.Product_Key.is_unique and d.Date.is_unique
assert c.Customer_Key.notna().all() and q.Product_Key.notna().all() and d.Date.notna().all()
assert f.Customer_Key.isin(c.Customer_Key).all()
assert f.Product_Key.isin(q.Product_Key).all()
assert f.Order_Date.isin(d.Date).all()
assert len(f)==len(s) and f.Sales.sum()==s.Sales.sum()
assert f.Order_ID.nunique()==s.Order_ID.nunique()
assert f.Quantity.sum()==s.Quantity.sum()
print('All primary/foreign key and reconciliation checks passed.')
print('Revenue:',f.Sales.sum(),'Orders:',f.Order_ID.nunique(),'Units:',f.Quantity.sum())
print('Customer names:',len(c),'Products:',len(q),'Calendar days:',len(d))
print('Profit unavailable; calendar dates do not imply observed transactions.')


All primary/foreign key and reconciliation checks passed.
Revenue: 81870 Orders: 30 Units: 90
Customer names: 6 Products: 4 Calendar days: 365
Profit unavailable; calendar dates do not imply observed transactions.
